# TrueCar Multi-City Download and Clean Dataset

This notebook downloads used-car listings from several diverse US metros, enriches listings with detail-page fields, and builds one cleaner modeling dataset.

Recommended workflow:

1. Start with `RUN_DOWNLOAD = False` and run the cleaning/report cells on existing files.
2. Set `RUN_DOWNLOAD = True` only when you want to refresh or expand the raw city files.
3. Use `data/truecar_clean_combined.csv` for regression/classification modeling.


In [ ]:
from pathlib import Path

import pandas as pd

from truecar_data_pipeline import (
    DEFAULT_CITIES,
    build_clean_dataset,
    scrape_cities,
    truecar_listing_url,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)


## Scrape Configuration

The default cities intentionally cover different used-car markets: Northeast, Southeast, Midwest, Texas, Mountain West, Southwest, Pacific Northwest, and California. The scraper excludes expanded-delivery inventory by default so each metro is more regionally meaningful.


In [ ]:
DATA_DIR = Path("data")

RUN_DOWNLOAD = False  # Change to True when you want to fetch fresh raw data.
MAX_PAGES_PER_CITY = 20
PAGE_SIZE = 100
SEARCH_RADIUS_MILES = 75
REQUEST_DELAY_SECONDS = 0.75

CITIES = DEFAULT_CITIES
CITIES


In [ ]:
# Quick sanity check for the URL pattern before a long scrape.
print(truecar_listing_url("Boston", "MA", page=1, page_size=PAGE_SIZE, search_radius=SEARCH_RADIUS_MILES))


## Download Raw City Files

This cell can take a while because it visits both search-result pages and individual detail pages. It writes one links file and one details file per metro under `data/`.


In [ ]:
if RUN_DOWNLOAD:
    scrape_results = scrape_cities(
        cities=CITIES,
        max_pages=MAX_PAGES_PER_CITY,
        page_size=PAGE_SIZE,
        search_radius=SEARCH_RADIUS_MILES,
        exclude_expanded_delivery=True,
        output_dir=DATA_DIR,
        delay_seconds=REQUEST_DELAY_SECONDS,
    )
    {city: df.shape for city, df in scrape_results.items()}
else:
    print("RUN_DOWNLOAD is False, so this notebook will use the raw CSV files already in data/.")


## Build Cleaner Combined Dataset

The cleaner fixes the main issues from the earlier notebooks: duplicate URLs/VINs, price parsing, odometer parsing, title splitting, dealer-location parsing, and feature-count creation.


In [ ]:
clean, report = build_clean_dataset(
    data_dir=DATA_DIR,
    output_csv=DATA_DIR / "truecar_clean_combined.csv",
    report_csv=DATA_DIR / "truecar_quality_report.csv",
)

print(clean.shape)
report


In [ ]:
clean.head(10)


## Market Coverage Checks

These quick summaries help decide whether you need more data from a particular region, make, or vehicle segment before modeling.


In [ ]:
metro_summary = (
    clean.groupby("source_metro", dropna=False)
    .agg(
        rows=("url", "count"),
        unique_vins=("vin", "nunique"),
        median_price=("sales_price", "median"),
        median_miles=("odometer_miles", "median"),
    )
    .sort_values("rows", ascending=False)
)
metro_summary


In [ ]:
make_summary = (
    clean.groupby("make", dropna=False)
    .agg(
        rows=("url", "count"),
        median_price=("sales_price", "median"),
        median_miles=("odometer_miles", "median"),
    )
    .sort_values("rows", ascending=False)
    .head(25)
)
make_summary


## Modeling Table

For regression, start with `sales_price` as the target. For classification, the notebook creates `price_tier`, which labels cars as `below_market`, `near_market`, or `above_market` using the listing price relative to TrueCar's average market price when available.


In [ ]:
model_columns = [
    "sales_price",
    "price_tier",
    "year",
    "vehicle_age",
    "make",
    "model",
    "trim",
    "odometer_miles",
    "fuel_type",
    "exterior",
    "interior",
    "dealer_state",
    "dealer_distance_miles",
    "source_metro",
    "feature_count",
    "standard_feature_count",
]

model_df = clean[[column for column in model_columns if column in clean.columns]].copy()
model_df.to_csv(DATA_DIR / "truecar_model_ready.csv", index=False)

print(model_df.shape)
model_df.head()


In [ ]:
# Classification target balance. If one class dominates, use stratified splits and consider class weights.
if "price_tier" in model_df:
    display(model_df["price_tier"].value_counts(dropna=False).to_frame("rows"))


## Output Files

After running the notebook, these files are the main artifacts:

- `data/truecar_clean_combined.csv`: richer deduped dataset with text/detail fields retained.
- `data/truecar_model_ready.csv`: smaller modeling table for regression/classification.
- `data/truecar_quality_report.csv`: basic data-quality counts.
